Install Libraries

In [ ]:
!pip pip install transformers torch scikit-learn

ERROR: unknown command "pip"


Define Library Imports

In [ ]:
import torch
import numpy as np
from google.colab import drive
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import re
import spacy
import glob
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from google.colab import drive
from textblob import TextBlob
from sentence_transformers import SentenceTransformer, util

Load Generated LLM Privacy Notices from GDrive

In [ ]:
# Authenticate to Google Drive to read in csv file
drive.mount('/content/drive')

# Specify folder path
folder_path = "/content/drive/MyDrive/{folder_name}/Datasets/**/*.txt"

# Get all .txt files
files = glob.glob(folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Read-in Raw Text From LLM Gen Privacy Notice

In [ ]:
# Create a method that opens and read the .txt files
# @param: filepath
def read_text(filepath):

    # Open and read .txt file contents
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()

    # Return contents
    return text

Segment Different Privacy Notices

In [ ]:
def get_all_privacy_notices(folder_path):
    path = Path(folder_path)
    notices_dict = {}

    # Use .rglob('*.txt') to search recursively the main directory and all subfolders
    for file_path in path.rglob('*.txt'):
        # Set key [filename]
        # value [filepath]
        notices_dict[file_path.name] = str(file_path.absolute())

    # Return dictionary
    return notices_dict

BERT Similarity or SpaCy Similarity Methods

In [ ]:
# Calculate the semantic similarity between two text files
# Returns the similarity score between 0.0 and 1.0
def check_file_similarity(file_path_a, file_path_b, method='transformer'):

  # Step 1: read file contents
  text_a = read_text(file_path_a)
  text_b = read_text(file_path_b)

  # Check method specified
  if method == 'transformer':

    # Define Sentence transformers (BERT-based)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Encode texts using BERT model
    embedding_a = model.encode(text_a, convert_to_tensor=True)
    embedding_b = model.encode(text_b, convert_to_tensor=True)

    # Calculate and return the cosine similarity
    similarity = util.cos_sim(embedding_a, embedding_b)

    return similarity.item()

  elif method == 'spacy':

    # Use spaCy word embeddings (original method)
    nlp = spacy.load("en_core_web_md")

    # Set max limit for text to be analyzed
    nlp.max_length = 1500000

    # Process the extracted text using the NLP model
    analyzed_text_a = nlp(text_a)
    analyzed_text_b = nlp(text_b)

    # Calculate and return the similarity seen
    return analyzed_text_a.similarity(analyzed_text_b)

  else:
    raise ValueError(f"Unknown method: {method}. Use 'transformer' or 'spacy'")

Compute Cosine Similarity

In [ ]:
def compute_similarity(privacy_notice_dict, models, app_name, personas, ref_file, method="transformer"):

  print(f"{app_name} Cosine Similarity")
  print("-" *40)

  # Store scores in dictionary
  scores = {}

  # Calculate and print similarity scores
  for persona_id, persona in personas:
    for model_id, display_name in models:
      file_key = f'{app_name}_{model_id}_{persona_id}_privacy_notice.txt'

      # Check if file exists in dictionary
      if file_key not in privacy_notice_dict:
          print(f"WARNING: {file_key} not found, skipping...")
          continue

      score = check_file_similarity(privacy_notice_dict[file_key], ref_file, method)
      print(f"{display_name} ({persona}) Similarity Score: {score:.3f}")

      # Store the computed scores
      scores[display_name] = score

  print("\n")
  return scores

Main Code Execution

In [ ]:
# Define the directory path
directory_path = "/content/drive/{folder_name}/Datasets/"

# Create dictionary of privacy notices
privacy_notice_dict = get_all_privacy_notices(directory_path)

# Define model identifiers and display names
models = [
    ('gemini-2.5-flash', 'Gemini 2.5 Flash'),
    ('gemini-2.5-pro', 'Gemini 2.5 Pro'),
    ('gpt-5', 'Chatgpt 5'),
    ('gpt-5.2', 'Chatgpt 5.2'),
    ('claude-haiku-4-5-20251001', 'Claude Haiku'),
    ('claude-sonnet-4-5-20250929', 'Claude Sonnet')
]

# Define app names
apps = [
    'Facebook', 'WhatsApp', 'YouTube', 'Instagram', 'TikTok', 'WeChat (Weixin)', 'Telegram', 'Messenger',
    'Snapchat', 'Viber', 'Reddit', 'Spotify', 'Pinterest', 'X (Twitter)', 'Shein', 'Quora', 'Teams',
    'LinkedIn', 'Vimeo', 'Threads', 'ShareChat', 'Twitch', 'Discord', 'Alibaba', 'Tumblr', 'AliExpress',
    'Medium', 'Google Maps', 'Uber', 'Foodpanda', 'DoorDash', 'Vinted', 'Yango', 'Wolt Delivery', 'Amazon',
    'Deliveroo', 'Glovo', 'HelloFresh', 'Just Eat', 'Temu', 'Gopuff', 'Uber Eats', 'Waze'
]

# Define persona
personas = [
    (1, 'GDPR Expert'),
    (2, 'Mobile Developer'),
    (3, 'Privacy Lawyer')
]

# Define similarity method: 'transformer' or 'spacy'
similarity_method = 'transformer'

print("="*60)
print(f"Using {similarity_method.upper()} method for Semantic Similarity")
print("="*60 + "\n")

# Create list to collect scores
similarity_scores = {}

# Compute Cosine Similarity using ground truth privacy notices for each app (except QQ)
for app in apps:

  ref_file = privacy_notice_dict[f'{app}_original_privacy_notice.txt']

  score = compute_similarity(privacy_notice_dict, models, app, personas, ref_file, method=similarity_method)

  # Add scores to dictionary
  similarity_scores[app] = score